In [1]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

print("Legacy Keras enabled:", os.environ["TF_USE_LEGACY_KERAS"])

Legacy Keras enabled: 1


In [3]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
import tensorflow_recommenders as tfrs

print("TFRS imported successfully")

In [ ]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

In [ ]:
import os

project_root = os.path.dirname(os.getcwd())
processed_path = os.path.join(project_root, "data", "processed")

print("Processed data path:")
print(processed_path)

print("\nFiles found:")
print(os.listdir(processed_path) if os.path.exists(processed_path) else "FOLDER DOES NOT EXIST")

In [ ]:
import pandas as pd

train_df = pd.read_csv(
    "../data/processed/train_interactions.csv"
)

validation_df = pd.read_csv(
    "../data/processed/validation_interactions.csv"
)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

display(train_df.head())

In [ ]:
print("=== TRAINING DATA ===")
print("Unique users:", train_df["customer_id"].nunique())
print("Unique items:", train_df["article_id"].nunique())

print("\n=== VALIDATION DATA ===")
print("Unique users:", validation_df["customer_id"].nunique())
print("Unique items:", validation_df["article_id"].nunique())

print("\n=== DATA TYPES ===")
print(train_df.dtypes)

print("\n=== MISSING VALUES ===")
print(train_df.isna().sum())
print(validation_df.isna().sum())

In [ ]:
import numpy as np

# Training vocabularies
user_ids = train_df["customer_id"].astype(str).unique()
item_ids = train_df["article_id"].astype(str).unique()

print("User vocabulary size:", len(user_ids))
print("Item vocabulary size:", len(item_ids))

print("\nSample user IDs:")
print(user_ids[:5])

print("\nSample item IDs:")
print(item_ids[:5])

In [ ]:
import tensorflow as tf

# Keep only the fields needed for the first Two-Tower retrieval model
train_data = train_df[["customer_id", "article_id"]].copy()
validation_data = validation_df[["customer_id", "article_id"]].copy()

# TensorFlow expects consistent string IDs
train_data["customer_id"] = train_data["customer_id"].astype(str)
train_data["article_id"] = train_data["article_id"].astype(str)

validation_data["customer_id"] = validation_data["customer_id"].astype(str)
validation_data["article_id"] = validation_data["article_id"].astype(str)

# Convert pandas dataframes to TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": train_data["customer_id"].values,
    "article_id": train_data["article_id"].values,
})

validation_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": validation_data["customer_id"].values,
    "article_id": validation_data["article_id"].values,
})

print("Training dataset:", train_ds)
print("Validation dataset:", validation_ds)

In [ ]:
BATCH_SIZE = 8192

train_ds = train_ds.shuffle(
    buffer_size=len(train_data),
    seed=42,
    reshuffle_each_iteration=True
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

validation_ds = validation_ds.batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

print("Batch size:", BATCH_SIZE)
print("Training dataset ready.")
print("Validation dataset ready.")

In [ ]:
# Final vocabulary lists for TensorFlow lookup layers

user_vocabulary = user_ids.tolist()
item_vocabulary = item_ids.tolist()

print("User vocabulary size:", len(user_vocabulary))
print("Item vocabulary size:", len(item_vocabulary))

In [ ]:
# ==============================
# USER TOWER
# ==============================

embedding_dimension = 64

user_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=user_vocabulary,
        mask_token=None
    ),
    tf.keras.layers.Embedding(
        input_dim=len(user_vocabulary) + 1,
        output_dim=embedding_dimension
    )
])

print("User tower created successfully.")
print("Embedding dimension:", embedding_dimension)

In [ ]:
# ==============================
# ITEM TOWER
# ==============================

item_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=item_vocabulary,
        mask_token=None
    ),
    tf.keras.layers.Embedding(
        input_dim=len(item_vocabulary) + 1,
        output_dim=embedding_dimension
    )
])

print("Item tower created successfully.")
print("Embedding dimension:", embedding_dimension)

In [ ]:
# ==============================
# ITEM CANDIDATE DATASET
# ==============================

item_ds = tf.data.Dataset.from_tensor_slices(
    item_vocabulary
)

item_ds = item_ds.batch(128).prefetch(tf.data.AUTOTUNE)

print("Item candidate dataset created.")
print("Candidate items:", len(item_vocabulary))

In [ ]:
# ==============================
# TWO-TOWER RETRIEVAL MODEL
# ==============================

class TwoTowerModel(tfrs.Model):

    def __init__(self, user_model, item_model):
        super().__init__()

        self.user_model = user_model
        self.item_model = item_model

        # Candidate dataset must contain article IDs as strings.
        candidate_dataset = (
            tf.data.Dataset.from_tensor_slices(item_vocabulary)
            .batch(128)
            .map(
                lambda article_id: self.item_model(article_id),
                num_parallel_calls=tf.data.AUTOTUNE
            )
            .prefetch(tf.data.AUTOTUNE)
        )

        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=candidate_dataset
            )
        )

    def compute_loss(self, features, training=False):
        user_embeddings = self.user_model(features["customer_id"])
        item_embeddings = self.item_model(features["article_id"])

        return self.task(
            user_embeddings,
            item_embeddings,
            compute_metrics=not training
        )


two_tower_model = TwoTowerModel(
    user_model,
    item_model
)

print("Two-Tower model recreated successfully.")

In [ ]:
# ==============================
# COMPILE TWO-TOWER MODEL
# ==============================
two_tower_model.compile(
    optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1)
)

print("Two-Tower model compiled successfully.")

In [ ]:
# ==============================
# TRAINING SMOKE TEST
# ==============================

history = two_tower_model.fit(
    train_ds.take(100),
    epochs=1
)

print("Smoke test training completed.")

In [ ]:
test_result = two_tower_model.evaluate(
    validation_ds.take(1),
    return_dict=True
)

print(test_result)

In [ ]:
# ==============================
# BASELINE TWO-TOWER TRAINING
# ==============================

history = two_tower_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=3
)

print("Baseline Two-Tower training completed.")

In [ ]:
# ==============================
# BASELINE METRIC SUMMARY
# ==============================

print("Baseline Two-Tower Results")
print("=" * 40)

for epoch in range(3):
    print(f"\nEpoch {epoch + 1}")
    print(
        "Val Top-1:  ",
        history.history["val_factorized_top_k/top_1_categorical_accuracy"][epoch]
    )
    print(
        "Val Top-5:  ",
        history.history["val_factorized_top_k/top_5_categorical_accuracy"][epoch]
    )
    print(
        "Val Top-10: ",
        history.history["val_factorized_top_k/top_10_categorical_accuracy"][epoch]
    )
    print(
        "Val Top-50: ",
        history.history["val_factorized_top_k/top_50_categorical_accuracy"][epoch]
    )
    print(
        "Val Top-100:",
        history.history["val_factorized_top_k/top_100_categorical_accuracy"][epoch]
    )
    print(
        "Val Loss:   ",
        history.history["val_loss"][epoch]
    ) 

In [ ]:
# ==============================
# INSPECT ENGINEERED FEATURES
# ==============================

user_features_path = os.path.join(
    processed_path,
    "enriched_user_features.csv"
)

item_features_path = os.path.join(
    processed_path,
    "enriched_item_features.csv"
)

user_features_df = pd.read_csv(user_features_path)
item_features_df = pd.read_csv(item_features_path)

print("User features shape:", user_features_df.shape)
print("Item features shape:", item_features_df.shape)

print("\nUser feature columns:")
print(user_features_df.columns.tolist())

print("\nItem feature columns:")
print(item_features_df.columns.tolist())

In [ ]:
# ==============================
# PREPARE CONTEXT FEATURES
# ==============================

user_numeric_features = [
    "age",
    "purchase_count",
    "avg_purchase_price",
    "total_spend"
]

item_numeric_features = [
    "product_type_no",
    "purchase_count",
    "avg_price",
    "total_revenue"
]

print("User numeric features:")
print(user_numeric_features)

print("\nItem numeric features:")
print(item_numeric_features)

In [ ]:
# ==============================
# NORMALIZE CONTEXT FEATURES
# ==============================

from sklearn.preprocessing import StandardScaler

user_scaler = StandardScaler()
item_scaler = StandardScaler()

user_features_df[user_numeric_features] = user_scaler.fit_transform(
    user_features_df[user_numeric_features].fillna(0)
)

item_features_df[item_numeric_features] = item_scaler.fit_transform(
    item_features_df[item_numeric_features].fillna(0)
)

print("User numeric features normalized.")
print("Item numeric features normalized.")

print("\nUser feature sample:")
print(user_features_df[user_numeric_features].head())

print("\nItem feature sample:")
print(item_features_df[item_numeric_features].head())

In [ ]:

# ==========================================
# BUILD CONTEXT-AWARE TRAINING AND VALIDATION DATA
# ==========================================

# Build user and item feature tables
user_context = user_features_df[
    [
        "customer_id",
        "age",
        "purchase_count",
        "avg_purchase_price",
        "total_spend",
    ]
].copy()

item_context = item_features_df[
    [
        "article_id",
        "product_type_no",
        "purchase_count",
        "avg_price",
        "total_revenue",
    ]
].copy()

user_context["customer_id"] = user_context["customer_id"].astype(str)
item_context["article_id"] = item_context["article_id"].astype(str)

# Prepare interaction data
train_base = train_df[["customer_id", "article_id"]].copy()
validation_base = validation_df[["customer_id", "article_id"]].copy()

for df in (train_base, validation_base):
    df["customer_id"] = df["customer_id"].astype(str)
    df["article_id"] = df["article_id"].astype(str)

# Join user and item features
context_train_df = (
    train_base
    .merge(user_context, on="customer_id", how="left")
    .merge(item_context, on="article_id", how="left")
)

context_validation_df = (
    validation_base
    .merge(user_context, on="customer_id", how="left")
    .merge(item_context, on="article_id", how="left")
)

print("Training rows:", len(context_train_df))
print("Validation rows:", len(context_validation_df))
print("Training shape:", context_train_df.shape)
print("Validation shape:", context_validation_df.shape)

print("\nTraining missing values:")
print(context_train_df.isna().sum())

print("\nValidation missing values:")
print(context_validation_df.isna().sum())


In [ ]:
print("Context training columns:")
print(context_train_df.columns.tolist())



In [ ]:
# Rename merged purchase-count columns
context_train_df = context_train_df.rename(
    columns={
        "purchase_count_x": "user_purchase_count",
        "purchase_count_y": "item_purchase_count"
    }
)

print("Context-aware training columns:")
print(context_train_df.columns.tolist())

print("\nMissing values:")
print(
    context_train_df[
        [
            "age",
            "user_purchase_count",
            "avg_purchase_price",
            "total_spend",
            "product_type_no",
            "item_purchase_count",
            "avg_price",
            "total_revenue"
        ]
    ].isna().sum()
)

In [ ]:
context_features = [
    "age",
    "user_purchase_count",
    "avg_purchase_price",
    "total_spend",
    "product_type_no",
    "item_purchase_count",
    "avg_price",
    "total_revenue"
]

context_train_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": context_train_df["customer_id"].astype(str).values,
    "article_id": context_train_df["article_id"].astype(str).values,
    "age": context_train_df["age"].astype("float32").values,
    "user_purchase_count": context_train_df["user_purchase_count"].astype("float32").values,
    "avg_purchase_price": context_train_df["avg_purchase_price"].astype("float32").values,
    "total_spend": context_train_df["total_spend"].astype("float32").values,
    "product_type_no": context_train_df["product_type_no"].astype("float32").values,
    "item_purchase_count": context_train_df["item_purchase_count"].astype("float32").values,
    "avg_price": context_train_df["avg_price"].astype("float32").values,
    "total_revenue": context_train_df["total_revenue"].astype("float32").values
})

context_train_ds = (
    context_train_ds
    .shuffle(100_000, seed=42)
    .batch(8192)
    .prefetch(tf.data.AUTOTUNE)
)

print("Context-aware TensorFlow dataset created.")

In [ ]:
context_batch = next(iter(context_train_ds))

print("Batch keys:")
print(context_batch.keys())

print("\nCustomer ID shape:", context_batch["customer_id"].shape)
print("Article ID shape:", context_batch["article_id"].shape)
print("Age shape:", context_batch["age"].shape)
print("Total spend shape:", context_batch["total_spend"].shape)

In [ ]:
class ContextUserTower(tf.keras.Model):

    def __init__(self, user_vocabulary, embedding_dimension=64):
        super().__init__()

        self.user_lookup = tf.keras.layers.StringLookup(
            vocabulary=user_vocabulary,
            mask_token=None
        )

        self.user_embedding = tf.keras.layers.Embedding(
            input_dim=len(user_vocabulary) + 1,
            output_dim=embedding_dimension
        )

        self.context_projection = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(embedding_dimension)
        ])

        self.normalization = tf.keras.layers.LayerNormalization()

    def call(self, inputs):
        user_ids = inputs["customer_id"]

        user_indices = self.user_lookup(user_ids)
        user_embedding = self.user_embedding(user_indices)

        context = tf.stack([
            inputs["age"],
            inputs["user_purchase_count"],
            inputs["avg_purchase_price"],
            inputs["total_spend"]
        ], axis=1)

        context_embedding = self.context_projection(context)

        combined = user_embedding + context_embedding

        return self.normalization(combined)


context_user_model = ContextUserTower(
    user_vocabulary=user_vocabulary,
    embedding_dimension=64
)

print("Context-aware User Tower created.")

In [ ]:
user_context_embeddings = context_user_model(context_batch)

print("User embedding shape:", user_context_embeddings.shape)

In [ ]:
class ContextItemTower(tf.keras.Model):

    def __init__(self, item_vocabulary, embedding_dimension=64):
        super().__init__()

        self.item_lookup = tf.keras.layers.StringLookup(
            vocabulary=item_vocabulary,
            mask_token=None
        )

        self.item_embedding = tf.keras.layers.Embedding(
            input_dim=len(item_vocabulary) + 1,
            output_dim=embedding_dimension
        )

        self.context_projection = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(embedding_dimension)
        ])

        self.normalization = tf.keras.layers.LayerNormalization()

    def call(self, inputs):
        item_ids = inputs["article_id"]

        item_indices = self.item_lookup(item_ids)
        item_embedding = self.item_embedding(item_indices)

        context = tf.stack([
            inputs["product_type_no"],
            inputs["item_purchase_count"],
            inputs["avg_price"],
            inputs["total_revenue"]
        ], axis=1)

        context_embedding = self.context_projection(context)

        combined = item_embedding + context_embedding

        return self.normalization(combined)


context_item_model = ContextItemTower(
    item_vocabulary=item_vocabulary,
    embedding_dimension=64
)

print("Context-aware Item Tower created.")

In [ ]:
item_context_embeddings = context_item_model(context_batch)

print("Item embedding shape:", item_context_embeddings.shape)

In [ ]:
class ContextAwareTwoTowerModel(tfrs.Model):

    def __init__(
        self,
        user_model,
        item_model,
        item_vocabulary
    ):
        super().__init__()

        self.user_model = user_model
        self.item_model = item_model

        candidate_dataset = (
            tf.data.Dataset.from_tensor_slices(item_vocabulary)
            .batch(128)
            .map(
                lambda article_id: self.item_model({
                    "article_id": article_id,
                    "product_type_no": tf.zeros(
                        tf.shape(article_id),
                        dtype=tf.float32
                    ),
                    "item_purchase_count": tf.zeros(
                        tf.shape(article_id),
                        dtype=tf.float32
                    ),
                    "avg_price": tf.zeros(
                        tf.shape(article_id),
                        dtype=tf.float32
                    ),
                    "total_revenue": tf.zeros(
                        tf.shape(article_id),
                        dtype=tf.float32
                    )
                }),
                num_parallel_calls=tf.data.AUTOTUNE
            )
            .prefetch(tf.data.AUTOTUNE)
        )

        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=candidate_dataset
            )
        )

    def compute_loss(self, features, training=False):

        user_embeddings = self.user_model(features)

        item_embeddings = self.item_model(features)

        return self.task(
            user_embeddings,
            item_embeddings,
            compute_metrics=not training
        )


context_two_tower_model = ContextAwareTwoTowerModel(
    user_model=context_user_model,
    item_model=context_item_model,
    item_vocabulary=item_vocabulary
)

context_two_tower_model.compile(
    optimizer=tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

print("Context-aware Two-Tower model created.")

In [ ]:
smoke_batch = next(iter(context_train_ds))

smoke_loss = context_two_tower_model.compute_loss(
    smoke_batch,
    training=True
)

print("Smoke-test loss:", float(smoke_loss))

In [ ]:
context_history = context_two_tower_model.fit(
    context_train_ds,
    epochs=3
)

print("Context-aware training completed.")

In [ ]:
context_validation_df = context_validation_df.rename(
    columns={
        "purchase_count_x": "user_purchase_count",
        "purchase_count_y": "item_purchase_count"
    }
)

print("Context-aware validation columns:")
print(context_validation_df.columns.tolist())

print("\nMissing values:")
print(
    context_validation_df[
        [
            "age",
            "user_purchase_count",
            "avg_purchase_price",
            "total_spend",
            "product_type_no",
            "item_purchase_count",
            "avg_price",
            "total_revenue"
        ]
    ].isna().sum()
)

In [ ]:
context_validation_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": context_validation_df["customer_id"].astype(str).values,
    "article_id": context_validation_df["article_id"].astype(str).values,
    "age": context_validation_df["age"].astype("float32").values,
    "user_purchase_count": context_validation_df["user_purchase_count"].astype("float32").values,
    "avg_purchase_price": context_validation_df["avg_purchase_price"].astype("float32").values,
    "total_spend": context_validation_df["total_spend"].astype("float32").values,
    "product_type_no": context_validation_df["product_type_no"].astype("float32").values,
    "item_purchase_count": context_validation_df["item_purchase_count"].astype("float32").values,
    "avg_price": context_validation_df["avg_price"].astype("float32").values,
    "total_revenue": context_validation_df["total_revenue"].astype("float32").values
})

context_validation_ds = (
    context_validation_ds
    .batch(8192)
    .prefetch(tf.data.AUTOTUNE)
)

print("Context-aware validation dataset created.")

In [ ]:
validation_batch = next(iter(context_validation_ds))

print("Validation batch keys:")
print(validation_batch.keys())

print("\nCustomer ID shape:", validation_batch["customer_id"].shape)
print("Article ID shape:", validation_batch["article_id"].shape)
print("Age shape:", validation_batch["age"].shape)

In [ ]:
context_eval_results = context_two_tower_model.evaluate(
    context_validation_ds,
    return_dict=True
)

print("Context-Aware Validation Results")
print("=" * 45)

for key, value in context_eval_results.items():
    print(f"{key}: {value}")

In [ ]:
baseline_metrics = {
    "Top-1": 0.00044288,
    "Top-5": 0.0015,
    "Top-10": 0.0019,
    "Top-50": 0.0047,
    "Top-100": 0.0076
}

context_metrics = {
    "Top-1": context_eval_results[
        "factorized_top_k/top_1_categorical_accuracy"
    ],
    "Top-5": context_eval_results[
        "factorized_top_k/top_5_categorical_accuracy"
    ],
    "Top-10": context_eval_results[
        "factorized_top_k/top_10_categorical_accuracy"
    ],
    "Top-50": context_eval_results[
        "factorized_top_k/top_50_categorical_accuracy"
    ],
    "Top-100": context_eval_results[
        "factorized_top_k/top_100_categorical_accuracy"
    ]
}

comparison_df = pd.DataFrame({
    "Metric": list(baseline_metrics.keys()),
    "Baseline": list(baseline_metrics.values()),
    "Context-Aware": [
        context_metrics[metric]
        for metric in baseline_metrics.keys()
    ]
})

comparison_df["Absolute Change"] = (
    comparison_df["Context-Aware"] -
    comparison_df["Baseline"]
)

comparison_df["Relative Change (%)"] = (
    comparison_df["Absolute Change"] /
    comparison_df["Baseline"]
) * 100

print(comparison_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

x = range(len(comparison_df))

plt.plot(
    x,
    comparison_df["Baseline"],
    marker="o",
    label="Baseline"
)

plt.plot(
    x,
    comparison_df["Context-Aware"],
    marker="o",
    label="Context-Aware"
)

plt.xticks(x, comparison_df["Metric"])
plt.xlabel("Retrieval Metric")
plt.ylabel("Categorical Accuracy")
plt.title("Baseline vs Context-Aware Two-Tower Model")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
comparison_path = os.path.join(
    processed_path,
    "two_tower_context_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print("Comparison saved to:")
print(comparison_path)

In [ ]:
model_path = os.path.join(
    project_root,
    "models",
    "two_tower"
)

os.makedirs(model_path, exist_ok=True)

print("Model directory:")
print(model_path)

In [ ]:
context_user_model.save(
    os.path.join(model_path, "context_user_tower.keras")
)

context_item_model.save(
    os.path.join(model_path, "context_item_tower.keras")
)

print("Context-aware towers saved successfully.")

In [ ]:
item_embedding_df = item_features_df[
    [
        "article_id",
        "product_type_no",
        "purchase_count",
        "avg_price",
        "total_revenue"
    ]
].copy()

item_embedding_df = item_embedding_df.rename(
    columns={
        "purchase_count": "item_purchase_count"
    }
)

item_embedding_ds = tf.data.Dataset.from_tensor_slices({
    "article_id": item_embedding_df["article_id"].astype(str).values,
    "product_type_no": item_embedding_df["product_type_no"].astype("float32").values,
    "item_purchase_count": item_embedding_df["item_purchase_count"].astype("float32").values,
    "avg_price": item_embedding_df["avg_price"].astype("float32").values,
    "total_revenue": item_embedding_df["total_revenue"].astype("float32").values
}).batch(4096)

print("Item embedding dataset created.")
print("Items:", len(item_embedding_df))

In [ ]:
item_embeddings = context_item_model.predict(
    item_embedding_ds,
    verbose=1
)

print("Item embeddings shape:")
print(item_embeddings.shape)

In [ ]:
item_embedding_path = os.path.join(
    model_path,
    "item_embeddings.npy"
)

item_id_path = os.path.join(
    model_path,
    "item_ids.npy"
)

np.save(item_embedding_path, item_embeddings)
np.save(
    item_id_path,
    item_embedding_df["article_id"].astype(str).values
)

print("Item embeddings saved.")
print(item_embedding_path)
print(item_id_path)

In [ ]:
user_embedding_df = user_features_df[
    [
        "customer_id",
        "age",
        "purchase_count",
        "avg_purchase_price",
        "total_spend"
    ]
].copy()

user_embedding_df = user_embedding_df.rename(
    columns={
        "purchase_count": "user_purchase_count"
    }
)

user_embedding_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": user_embedding_df["customer_id"].astype(str).values,
    "age": user_embedding_df["age"].astype("float32").values,
    "user_purchase_count": user_embedding_df["user_purchase_count"].astype("float32").values,
    "avg_purchase_price": user_embedding_df["avg_purchase_price"].astype("float32").values,
    "total_spend": user_embedding_df["total_spend"].astype("float32").values
}).batch(4096)

print("User embedding dataset created.")
print("Users:", len(user_embedding_df))

In [ ]:
user_embeddings = context_user_model.predict(
    user_embedding_ds,
    verbose=1
)

print("User embeddings shape:")
print(user_embeddings.shape)

In [ ]:
user_embedding_path = os.path.join(
    model_path,
    "user_embeddings.npy"
)

user_id_path = os.path.join(
    model_path,
    "user_ids.npy"
)

np.save(user_embedding_path, user_embeddings)

np.save(
    user_id_path,
    user_embedding_df["customer_id"].astype(str).values
)

print("User embeddings saved.")
print(user_embedding_path)
print(user_id_path)

In [ ]:
print("Saved Two-Tower artifacts:")
print("=" * 40)

for filename in os.listdir(model_path):
    filepath = os.path.join(model_path, filename)
    
    if os.path.isfile(filepath):
        size_mb = os.path.getsize(filepath) / (1024 ** 2)
        print(f"{filename:35s} {size_mb:8.2f} MB")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Select one known user
test_user_index = 0

test_user_id = user_embedding_df.iloc[test_user_index]["customer_id"]
test_user_embedding = user_embeddings[test_user_index].reshape(1, -1)

# Compare against all item embeddings
similarities = cosine_similarity(
    test_user_embedding,
    item_embeddings
)[0]

# Get top 10 items
top_indices = np.argsort(similarities)[::-1][:10]

retrieved_items = pd.DataFrame({
    "article_id": item_embedding_df.iloc[top_indices]["article_id"].values,
    "similarity": similarities[top_indices]
})

print("Test User:")
print(test_user_id)

print("\nTop-10 Retrieved Items:")
print(retrieved_items.to_string(index=False))

In [ ]:
print("Two-Tower Training Summary")
print("=" * 50)

print("\nBaseline Validation:")
for metric, value in baseline_metrics.items():
    print(f"{metric:8s}: {value:.6f}")

print("\nContext-Aware Validation:")
for metric, value in context_metrics.items():
    print(f"{metric:8s}: {value:.6f}")

print("\nModel Configuration:")
print("Embedding dimension: 64")
print("Training interactions:", len(train_df))
print("Validation interactions:", len(validation_df))
print("User vocabulary:", len(user_vocabulary))
print("Item vocabulary:", len(item_vocabulary))

print("\nArtifacts:")
print("User tower: context_user_tower.keras")
print("Item tower: context_item_tower.keras")
print("User embeddings: user_embeddings.npy")
print("Item embeddings: item_embeddings.npy")

print("\nNotebook status: COMPLETE")